# Question Answering with Transformers

**Objective:** extract an answer from a supplied passage and question using a transformer already fine-tuned for question answering, then evaluate it on SQuAD v1.1 with Exact Match (EM) and token F1.

This notebook uses `distilbert-base-cased-distilled-squad`, a checkpoint already fine-tuned on SQuAD v1.1. It performs inference and evaluation; it does **not** perform additional fine-tuning. The training split is inspected only, and scoring uses validation examples.


## 1. Setup


In [1]:
import json
import random
import re
import string
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from transformers.utils import logging as transformers_logging

SEED = 42
MODEL_CHECKPOINT = "distilbert-base-cased-distilled-squad"
MODEL_REVISION = "564e9b582944a57a3e586bbb98fd6f0a4118db7f"
EVAL_SIZE = None  # Full validation; use 1000 for a faster random-sample run.
BATCH_SIZE = 8
MAX_LENGTH = 384
STRIDE = 128
MAX_ANSWER_LENGTH = 30  # Maximum predicted answer length in tokens.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
transformers_logging.disable_progress_bar()

print(f"Device: {device}; seed: {SEED}")
display(pd.DataFrame({
    "Package": ["torch", "transformers", "numpy", "pandas"],
    "Version": [version(name) for name in ["torch", "transformers", "numpy", "pandas"]],
}))

Device: cuda; seed: 42


,Package,Version
0,torch,2.10.0+cu126
1,transformers,5.17.0
2,numpy,2.4.6
3,pandas,3.0.1


## 2. Load and inspect SQuAD v1.1

Each example contains a context, a question, and reference answers with character offsets. Validation questions can have several valid reference answers. Evaluation gives each prediction its best score against these references.

In [ ]:
def load_squad_json(path):
    with path.open(encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("version") != "1.1":
        raise ValueError(f"Expected SQuAD v1.1: {path}")
    return [
        {
            "id": question["id"],
            "title": article["title"],
            "context": paragraph["context"],
            "question": question["question"],
            "answers": {
                "text": [answer["text"] for answer in question["answers"]],
                "answer_start": [answer["answer_start"] for answer in question["answers"]],
            },
        }
        for article in payload["data"]
        for paragraph in article["paragraphs"]
        for question in paragraph["qas"]
    ]


data_dir = next((
    path for path in [Path("archive")]
    if (path / "train-v1.1.json").is_file() and (path / "dev-v1.1.json").is_file()
), None)

if data_dir is not None:
    raw_datasets = {
        "train": load_squad_json(data_dir / "train-v1.1.json"),
        "validation": load_squad_json(data_dir / "dev-v1.1.json"),
    }

    dataset_source = f"Local SQuAD v1.1 JSON: {data_dir}"
    
else:
    from datasets import load_dataset
    raw_datasets = load_dataset("rajpurkar/squad")
    dataset_source = "Hugging Face: rajpurkar/squad"

print(dataset_source)
display(pd.DataFrame([
    {"Split": split, "Questions": len(examples)}
    for split, examples in raw_datasets.items()
]))

example = raw_datasets["validation"][0]
display(pd.DataFrame([{
    "Question": example["question"],
    "Reference answers": example["answers"]["text"],
}]))

Hugging Face: rajpurkar/squad


,Split,Questions
0,train,87599
1,validation,10570


,Question,Reference answers
0,Which NFL team represented the AFC at Super Bo...,"[Denver Broncos, Denver Broncos, Denver Broncos]"


## 3. Load the QA model and prepare context windows

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT, revision=MODEL_REVISION, use_fast=True
)
model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_CHECKPOINT, revision=MODEL_REVISION
).to(device)
model.eval()
assert tokenizer.is_fast, "Character-offset extraction requires a fast tokenizer."


def preprocess_examples(examples):
    encoded = tokenizer(
        [example["question"].strip() for example in examples],
        [example["context"] for example in examples],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = encoded.pop("overflow_to_sample_mapping")
    offsets = encoded.pop("offset_mapping")
    context_offsets = [
        [offset if sequence_id == 1 else None
         for offset, sequence_id in zip(window_offsets, encoded.sequence_ids(i))]
        for i, window_offsets in enumerate(offsets)
    ]
    model_inputs = {
        name: torch.tensor(values, dtype=torch.long)
        for name, values in encoded.items()
        if name in tokenizer.model_input_names
    }
    return model_inputs, sample_map, context_offsets


print(f"Model: {MODEL_CHECKPOINT}")
print(f"Revision: {MODEL_REVISION}")

Model: distilbert-base-cased-distilled-squad
Revision: 564e9b582944a57a3e586bbb98fd6f0a4118db7f


## 4. Predict and extract answer spans


In [ ]:
def decode_window(start_logits, end_logits, offsets):
    best = None
    for start_index, start_offset in enumerate(offsets):
        if start_offset is None:
            continue
        for end_index in range(start_index, min(len(offsets), start_index + MAX_ANSWER_LENGTH)):
            end_offset = offsets[end_index]
            if end_offset is None or end_offset[1] <= start_offset[0]:
                continue
            score = float(start_logits[start_index] + end_logits[end_index])
            if best is None or score > best["span_score"]:
                best = {
                    "start": start_offset[0],
                    "end": end_offset[1],
                    "span_score": score,
                }
    return best


def predict_answers(examples):
    if not examples:
        return []
    if any(not example["question"].strip() or not example["context"].strip()
           for example in examples):
        raise ValueError("Each question and context must be nonempty.")
    
    model_inputs, sample_map, offsets = preprocess_examples(examples)
    best_answers = [None] * len(examples)

    with torch.inference_mode():
        for batch_start in range(0, len(sample_map), BATCH_SIZE):
            batch_end = batch_start + BATCH_SIZE
            batch = {name: values[batch_start:batch_end].to(device)
                     for name, values in model_inputs.items()}
            
            outputs = model(**batch)
            starts = outputs.start_logits.cpu().numpy()
            ends = outputs.end_logits.cpu().numpy()
            
            for row, feature_index in enumerate(range(batch_start, min(batch_end, len(sample_map)))):
                candidate = decode_window(starts[row], ends[row], offsets[feature_index])
                example_index = sample_map[feature_index]
                previous = best_answers[example_index]
                if candidate is not None and (previous is None or candidate["span_score"] > previous["span_score"]):
                    best_answers[example_index] = candidate

    predictions = []
    for example, answer in zip(examples, best_answers):
        if answer is None:
            raise ValueError(f"No valid context span for example {example['id']}")
        
        predictions.append({
            "id": example["id"],
            "prediction_text": example["context"][answer["start"]:answer["end"]],
            **answer,
        })
        
    return predictions


def answer_question(question, context):
    """Return an answer and its character offsets for a supplied passage."""
    return predict_answers([{"id": "custom", "question": question, "context": context}])[0]


context = (
    "Marie Curie was born in Warsaw in 1867. "
    "She won the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911."
)
question = "Where was Marie Curie born?"
answer = answer_question(question, context)
print(f"Context: {context}")
print(f"Question: {question}")
print(f"Answer: {answer['prediction_text']} (characters {answer['start']}:{answer['end']})")

Context: Marie Curie was born in Warsaw in 1867. She won the Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911.
Question: Where was Marie Curie born?
Answer: Warsaw (characters 24:30)


## 5. Evaluate Exact Match and F1

The metrics follow [the SQuAD v1.1 scorer maintained in Hugging Face Evaluate](https://github.com/huggingface/evaluate/blob/main/metrics/squad/compute_score.py): lowercase text, remove punctuation and English articles, and normalize whitespace. **EM** measures normalized string equality; **F1** measures token overlap, including repeated-token counts. Each metric takes the maximum across a question's reference answers and averages over questions, on a 0–100 scale.


In [5]:
def normalize_answer(text):
    text = "".join(character for character in text.lower() if character not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, reference):
    return float(normalize_answer(prediction) == normalize_answer(reference))


def token_f1(prediction, reference):
    predicted_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()
    overlap = sum((Counter(predicted_tokens) & Counter(reference_tokens)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(predicted_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def score_predictions(examples, predictions):
    predictions_by_id = {prediction["id"]: prediction for prediction in predictions}
    expected_ids = {example["id"] for example in examples}
    if (len(expected_ids) != len(examples) or len(predictions_by_id) != len(predictions)
            or set(predictions_by_id) != expected_ids):
        raise ValueError("Require exactly one prediction per unique evaluation question.")
    rows = []
    for example in examples:
        prediction = predictions_by_id[example["id"]]["prediction_text"]
        references = example["answers"]["text"]
        rows.append({
            "id": example["id"],
            "question": example["question"],
            "prediction": prediction,
            "references": references,
            "exact_match": max(exact_match(prediction, reference) for reference in references),
            "f1": max(token_f1(prediction, reference) for reference in references),
        })
    return pd.DataFrame(rows)


validation_examples = list(raw_datasets["validation"])
if EVAL_SIZE is None:
    eval_examples = validation_examples
elif isinstance(EVAL_SIZE, int) and 0 < EVAL_SIZE <= len(validation_examples):
    eval_examples = random.Random(SEED).sample(validation_examples, EVAL_SIZE)
else:
    raise ValueError("EVAL_SIZE must be None or a positive integer within the validation split.")

started = time.perf_counter()
predictions = predict_answers(eval_examples)
elapsed = time.perf_counter() - started
scores = score_predictions(eval_examples, predictions)
metrics = {
    "Evaluation scope": "Full validation" if len(eval_examples) == len(validation_examples) else "Random validation sample",
    "Questions": len(eval_examples),
    "Seed": SEED,
    "Exact Match (%)": 100 * scores["exact_match"].mean(),
    "F1 (%)": 100 * scores["f1"].mean(),
    "Inference seconds": elapsed,
}
display(pd.DataFrame([metrics]).round(2))

,Evaluation scope,Questions,Seed,Exact Match (%),F1 (%),Inference seconds
0,Full validation,10570,42,79.44,86.8,83.99


## 6. Inspect predictions and summarize results

The tables show up to three exact matches and three non-exact matches in evaluation order.

In [6]:
columns = ["question", "prediction", "references", "exact_match", "f1"]
with pd.option_context("display.max_colwidth", 110):
    display(Markdown("**Exact matches**"))
    display(scores.loc[scores["exact_match"] == 1, columns].head(3).round(3))
    display(Markdown("**Non-exact matches**"))
    display(scores.loc[scores["exact_match"] == 0, columns].head(3).round(3))

display(Markdown(
    f"On **{len(eval_examples):,} validation questions** "
    f"({metrics['Evaluation scope'].lower()}), the model achieved "
    f"**{metrics['Exact Match (%)']:.2f}% EM** and **{metrics['F1 (%)']:.2f}% F1**. "
    "These are measured results from the execution above. "
    "A higher F1 than EM means some predictions overlap the reference answer without matching it exactly."
))

**Exact matches**

,question,prediction,references,exact_match,f1
0,Which NFL team represented the AFC at Super Bowl 50?,Denver Broncos,"[Denver Broncos, Denver Broncos, Denver Broncos]",1.0,1.0
1,Which NFL team represented the NFC at Super Bowl 50?,Carolina Panthers,"[Carolina Panthers, Carolina Panthers, Carolina Panthers]",1.0,1.0
2,Where did Super Bowl 50 take place?,"Levi's Stadium in the San Francisco Bay Area at Santa Clara, California","[Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, Cal...",1.0,1.0


**Non-exact matches**

,question,prediction,references,exact_match,f1
3,Which NFL team won Super Bowl 50?,Carolina Panthers,"[Denver Broncos, Denver Broncos, Denver Broncos]",0.0,0.00
7,What is the AFC short for?,Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL...,"[American Football Conference, American Football Conference, American Football Conference]",0.0,0.25
13,What city did Super Bowl 50 take place in?,"Santa Clara, California","[Santa Clara, Santa Clara, Santa Clara]",0.0,0.80


On **10,570 validation questions** (full validation), the model achieved **79.44% EM** and **86.80% F1**. These are measured results from the execution above. A higher F1 than EM means some predictions overlap the reference answer without matching it exactly.